# Global Ecommerce Sales Data Cleaning

This notebook loads a raw ecommerce sales dataset, performs cleaning and validation, engineers new features, and saves the cleaned dataset to a new file.

In [1]:
import pandas as pd
import numpy as np
import os

## 0. Define file paths

Set the input file path, output directory, and cleaned output file path.  
Create the output directory if it does not already exist.

In [2]:
raw_file_path = r"global_ecommerce_sales.csv"
output_dir = r"../Datasets/Cleaned/global_ecommerce"
output_file_path = os.path.join(output_dir, "global_ecommerce_sales_cleaned.csv")

os.makedirs(output_dir, exist_ok=True)

## 1. Load the data

Read the raw CSV file into a pandas DataFrame and inspect its structure.

In [3]:
df = pd.read_csv(raw_file_path)

print("Initial shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nDataset Info:")
print(df.info())

Initial shape: (2000, 15)

Columns:
['Order_ID', 'Order_Date', 'Customer_Name', 'Customer_Segment', 'Country', 'Region', 'Product_Category', 'Product_Name', 'Quantity', 'Unit_Price', 'Discount_Percent', 'Total_Sales', 'Shipping_Cost', 'Profit', 'Payment_Method']

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Order_ID          2000 non-null   object 
 1   Order_Date        2000 non-null   object 
 2   Customer_Name     2000 non-null   object 
 3   Customer_Segment  2000 non-null   object 
 4   Country           2000 non-null   object 
 5   Region            2000 non-null   object 
 6   Product_Category  2000 non-null   object 
 7   Product_Name      2000 non-null   object 
 8   Quantity          2000 non-null   int64  
 9   Unit_Price        2000 non-null   float64
 10  Discount_Percent  2000 non-null   int64  
 11 

## 2. Convert data types

Convert the order date to datetime format, numeric fields to numeric values,  
and standardize text-based columns by trimming spaces and formatting categories.

In [4]:
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")

numeric_cols = [
    "Quantity",
    "Unit_Price",
    "Discount_Percent",
    "Total_Sales",
    "Shipping_Cost",
    "Profit"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

text_cols = [
    "Order_ID",
    "Customer_Name",
    "Customer_Segment",
    "Country",
    "Region",
    "Product_Category",
    "Product_Name",
    "Payment_Method"
]

for col in text_cols:
    df[col] = df[col].str.strip()

categorical_cols = [
    "Customer_Segment",
    "Country",
    "Region",
    "Product_Category",
    "Payment_Method"
]

for col in categorical_cols:
    df[col] = df[col].str.title()

## 3. Check missing values

Summarize missing values across all columns to identify gaps in the dataset.

In [5]:
missing_summary = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percent": df.isnull().mean() * 100
}).sort_values(by="Missing_Count", ascending=False)

print("\nMissing values summary:")
print(missing_summary)


Missing values summary:
                  Missing_Count  Missing_Percent
Order_ID                      0              0.0
Order_Date                    0              0.0
Customer_Name                 0              0.0
Customer_Segment              0              0.0
Country                       0              0.0
Region                        0              0.0
Product_Category              0              0.0
Product_Name                  0              0.0
Quantity                      0              0.0
Unit_Price                    0              0.0
Discount_Percent              0              0.0
Total_Sales                   0              0.0
Shipping_Cost                 0              0.0
Profit                        0              0.0
Payment_Method                0              0.0


## 4. Remove exact duplicates

Check for and remove rows that are exact duplicates.

In [6]:
duplicate_count = df.duplicated().sum()
print("\nExact duplicate rows:", duplicate_count)

df = df.drop_duplicates()


Exact duplicate rows: 0


## 5. Validate business rules

Apply simple data-quality rules to flag suspicious values:

- Quantity should be between 1 and 15
- Order date should be between 2023-01-01 and 2025-12-31
- Discount percent should be between 0 and 30

In [7]:
invalid_quantity = df[(df["Quantity"] < 1) | (df["Quantity"] > 15)]
print("\nInvalid quantity rows:", len(invalid_quantity))

invalid_dates = df[
    (df["Order_Date"] < "2023-01-01") |
    (df["Order_Date"] > "2025-12-31")
]
print("Invalid date rows:", len(invalid_dates))

invalid_discount = df[(df["Discount_Percent"] < 0) | (df["Discount_Percent"] > 30)]
print("Invalid discount rows:", len(invalid_discount))


Invalid quantity rows: 0
Invalid date rows: 0
Invalid discount rows: 0


## 6. Validate and recalculate total sales

Recompute expected total sales from quantity, unit price, and discount.  
Compare against the recorded values and overwrite `Total_Sales` with the calculated value.

In [8]:
df["Expected_Total_Sales"] = (
    df["Quantity"] * df["Unit_Price"] * (1 - df["Discount_Percent"] / 100)
).round(2)

df["Recorded_Total_Sales"] = df["Total_Sales"].round(2)

df["Sales_Match"] = np.isclose(
    df["Recorded_Total_Sales"],
    df["Expected_Total_Sales"],
    atol=0.01
)

mismatch_count = (~df["Sales_Match"]).sum()
print("Sales mismatches:", mismatch_count)

df["Total_Sales"] = df["Expected_Total_Sales"]

Sales mismatches: 0


## 7. Handle missing values

Drop rows missing critical financial fields and fill missing text categories with `"Unknown"`.

In [9]:
df = df.dropna(subset=["Order_Date", "Quantity", "Unit_Price", "Total_Sales", "Profit"])

for col in categorical_cols + ["Product_Name", "Customer_Name"]:
    df[col] = df[col].fillna("Unknown")

## 8. Feature engineering

Create time-based features and financial ratio features:

- Year
- Month
- Month name
- Quarter
- Profit margin
- Shipping cost ratio

In [10]:
df["Year"] = df["Order_Date"].dt.year
df["Month"] = df["Order_Date"].dt.month
df["Month_Name"] = df["Order_Date"].dt.month_name()
df["Quarter"] = df["Order_Date"].dt.quarter

df["Profit_Margin"] = np.where(
    df["Total_Sales"] != 0,
    df["Profit"] / df["Total_Sales"],
    np.nan
)

df["Shipping_Cost_Ratio"] = np.where(
    df["Total_Sales"] != 0,
    df["Shipping_Cost"] / df["Total_Sales"],
    np.nan
)

## 9. Final validation checks

Review the cleaned dataset structure, missing values, date range, negative profit rows,  
and the number of unique values in key categorical fields.

In [11]:
print("\nFinal shape:", df.shape)

print("\nRemaining missing values:")
print(df.isnull().sum())

print("\nDate range after cleaning:")
print(df["Order_Date"].min(), "to", df["Order_Date"].max())

print("\nNegative profit rows:", (df["Profit"] < 0).sum())

print("\nUnique category counts:")
for col in categorical_cols:
    print(f"{col}: {df[col].nunique()} unique values")


Final shape: (2000, 24)

Remaining missing values:
Order_ID                0
Order_Date              0
Customer_Name           0
Customer_Segment        0
Country                 0
Region                  0
Product_Category        0
Product_Name            0
Quantity                0
Unit_Price              0
Discount_Percent        0
Total_Sales             0
Shipping_Cost           0
Profit                  0
Payment_Method          0
Expected_Total_Sales    0
Recorded_Total_Sales    0
Sales_Match             0
Year                    0
Month                   0
Month_Name              0
Quarter                 0
Profit_Margin           0
Shipping_Cost_Ratio     0
dtype: int64

Date range after cleaning:
2023-01-02 00:00:00 to 2025-12-31 00:00:00

Negative profit rows: 272

Unique category counts:
Customer_Segment: 3 unique values
Country: 20 unique values
Region: 5 unique values
Product_Category: 4 unique values
Payment_Method: 4 unique values


## 10. Save cleaned dataset

Write the cleaned DataFrame to a CSV file in the output directory.

In [12]:
df.to_csv(output_file_path, index=False)

print(f"\nCleaned dataset saved to:\n{output_file_path}")


Cleaned dataset saved to:
../Datasets/Cleaned/global_ecommerce\global_ecommerce_sales_cleaned.csv
